# Загрузка данных

In [ ]:
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_m2_geo = pd.read_csv ('data_for_models/df_m2_geo.csv', low_memory=False)

In [ ]:
df_m2_geo.info()

# Catboost M2.1

In [ ]:
groups = df_m2_geo['region_name']

In [ ]:
results = []

In [ ]:
gkf = GroupKFold(n_splits=5)

In [ ]:
models_config = {
    "M2.1.0_structure": {  # baseline: только структурные признаки
        "cat": ['role_name', 'schedule_id', 'employment_id','experience_ord'],
        "num": []
    }
    ,
    "M2.1.1_region": {  # добавляем экономический регион, регион и город
        "cat": ['role_name', 'schedule_id', 'employment_id', 'economic_region', 'region_name', 'address.city_new','experience_ord'],
        "num": []
    }
    ,
    "M2.1.2_sincos": {  # добавляем синусы/косинусы координат
        "cat": ['role_name', 'schedule_id', 'employment_id', 'economic_region', 'region_name', 'address.city_new','experience_ord'],
        "num": ['lat_sin', 'lat_cos', 'lon_sin', 'lon_cos', 'geo_available']
    }
    ,
    "M2.1.3_geohash": {  # добавляем geohash
        "cat": ['role_name', 'schedule_id', 'employment_id', 'economic_region', 'region_name', 'address.city_new', 'experience_ord',
          'geohash_4', 'geohash_5', 'geohash_6'],
        "num": [ 'lat_sin', 'lat_cos', 'lon_sin', 'lon_cos','geo_available']
    }
    ,
    "M2.1.4_distance": {  # добавляем distance_to_reg_center и distance_missing
        "cat": ['role_name', 'schedule_id', 'employment_id', 'economic_region', 'region_name', 'address.city_new', 'experience_ord',
          'geohash_4', 'geohash_5', 'geohash_6'],
        "num": [ 'lat_sin', 'lat_cos', 'lon_sin', 'lon_cos','geo_available','distance_to_reg_center', 'distance_missing']
    }
}

In [ ]:
catboost_params = {
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "RMSE",
    "task_type": "GPU",
    "devices": "0",
    "gpu_ram_part": 0.8,
    "random_seed": 42,
    "verbose": 100
}

In [ ]:
N_FOLDS = int(gkf.n_splits)
for model_name, config in tqdm(models_config.items(), desc="Models", position=0):
    print(f"\nTraining {model_name}...")
    features = config["cat"] + config["num"]
    X = df_m2_geo[features]
    y = df_m2_geo["salary_from_log"]
    rmse_scores, mae_scores, r2_scores = [], [], []
    for fold_idx, (train_idx, val_idx) in enumerate(
        tqdm(
            gkf.split(X, y, groups),
            desc=f"Folds ({model_name})",
            position=1,
            leave=False,
        )
    ):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        model = CatBoostRegressor(**catboost_params)
        model.fit(
            X_train,
            y_train,
            cat_features=config["cat"],
            eval_set=(X_val, y_val),
            verbose=0,
        )
        preds = model.predict(X_val)
        rmse_scores.append(np.sqrt(mean_squared_error(y_val, preds)))
        mae_scores.append(mean_absolute_error(y_val, preds))
        r2_scores.append(r2_score(y_val, preds))
    results.append(
        {
            "model": model_name,
            "CV_RMSE_mean": float(np.mean(rmse_scores)),
            "CV_RMSE_std": float(np.std(rmse_scores)),
            "CV_MAE_mean": float(np.mean(mae_scores)),
            "CV_MAE_std": float(np.std(mae_scores)),
            "CV_R2_mean": float(np.mean(r2_scores)),
            "CV_R2_std": float(np.std(r2_scores)),
            "n_folds": N_FOLDS,
        }
    )


In [ ]:
# Быстрая проверка, что в results действительно новые поля CV_*
results_df_check = pd.DataFrame(results)
required_cols = {
    "model", "CV_RMSE_mean", "CV_RMSE_std",
    "CV_MAE_mean", "CV_MAE_std",
    "CV_R2_mean", "CV_R2_std", "n_folds"
}
missing_cols = required_cols - set(results_df_check.columns)
assert not missing_cols, f"В results отсутствуют колонки: {missing_cols}"
display(results_df_check.head())

# Анализ метрик

In [ ]:
results_df = pd.DataFrame(results).copy()

In [ ]:
results_df_geo = pd.DataFrame(results).sort_values("CV_RMSE_mean")
results_df_geo

In [ ]:
results_df = pd.DataFrame(results)
baseline_rmse = results_df.loc[
    results_df["model"] == "M2.1.0_structure", "CV_RMSE_mean"
].values[0]
baseline_r2 = results_df.loc[
    results_df["model"] == "M2.1.0_structure", "CV_R2_mean"
].values[0]
results_df["RMSE_improvement_%"] = (
    baseline_rmse - results_df["CV_RMSE_mean"]
) / baseline_rmse * 100
results_df["R2_gain"] = results_df["CV_R2_mean"] - baseline_r2
results_df = results_df.sort_values("model")
display(results_df.round(6))

In [ ]:
#Сохраним метрики

metrics_dir = Path("data_for_models")
metrics_dir.mkdir(parents=True, exist_ok=True)
m2_cv_metrics = results_df.copy()
preferred_cols = [
    "model",
    "CV_RMSE_mean", "CV_RMSE_std",
    "CV_MAE_mean", "CV_MAE_std",
    "CV_R2_mean", "CV_R2_std",
    "n_folds",
    "RMSE_improvement_%", "R2_gain"
]
m2_cv_metrics = m2_cv_metrics[[c for c in preferred_cols if c in m2_cv_metrics.columns]]
m2_cv_metrics = m2_cv_metrics.rename(columns={"model": "Model"})
m2_cv_metrics_path = metrics_dir / "m2_cv_metrics.csv"
m2_cv_metrics.to_csv(m2_cv_metrics_path, index=False, encoding="utf-8-sig")
print(f"Saved: {m2_cv_metrics_path}")
display(m2_cv_metrics.round(6))

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(results_df["model"], results_df["CV_R2_mean"], marker="o")
plt.title("Model performance improvement with spatial features")
plt.xlabel("Model stage")
plt.ylabel("CV_R2_mean")

plt.grid(True)
plt.show()

In [ ]:
# Выберем лучшую конфигурацию по CV_RMSE_mean (меньше = лучше)
results_df = pd.DataFrame(results).copy()
best_row = results_df.loc[results_df["CV_RMSE_mean"].idxmin()]
best_model_name = best_row["model"]
best_config = models_config[best_model_name]
best_features = best_config["cat"] + best_config["num"]
print(f"Best model by CV_RMSE_mean: {best_model_name}")
display(best_row.to_frame().T)

In [ ]:
# Сделаем финальный refit выбранной лучшей конфигурации на полной выборке
X_final = df_m2_geo[best_features]
y_final = df_m2_geo["salary_from_log"]
best_model = CatBoostRegressor(**catboost_params)
best_model.fit(X_final, y_final, cat_features=best_config["cat"], verbose=0)
y_pred = best_model.predict(X_final)
rmse_best = np.sqrt(mean_squared_error(y_final, y_pred))
mae_best = mean_absolute_error(y_final, y_pred)
r2_best = r2_score(y_final, y_pred)
best_model_metrics = pd.DataFrame([{
    "Model": f"{best_model_name} (best)",
    "RMSE": rmse_best,
    "MAE": mae_best,
    "R2": r2_best
}])
print(best_model_metrics.round(4))

# Feature Importance и SHAP

In [ ]:
#Строим график feature_importance

feature_importance = best_model.get_feature_importance(type='FeatureImportance')

fi_df = pd.DataFrame({
    'feature': best_model.feature_names_,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 6))

sns.barplot(
    data=fi_df,
    x='importance',
    y='feature',
    palette='viridis'
)

plt.title(f'Feature Importance - {best_model_name}', fontsize=14)
plt.xlabel('Importance')
plt.ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
pool = Pool(
    X_final,
    cat_features=best_config["cat"]
)

shap_values = best_model.get_feature_importance(
    pool,
    type='ShapValues'
)

# убираем bias (последний столбец)
shap_values = shap_values[:, :-1]

In [ ]:
shap_df = pd.DataFrame(
    np.abs(shap_values).mean(axis=0),
    index=best_model.feature_names_,
    columns=['mean_abs_shap']
).sort_values('mean_abs_shap', ascending=False)

plt.figure(figsize=(9, 6))

sns.barplot(
    data=shap_df.reset_index(),
    x='mean_abs_shap',
    y='index',
    palette='coolwarm'
)

plt.title(F'Mean |SHAP value| — {best_model_name}', fontsize=14)
plt.xlabel('Mean absolute SHAP value')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
fi_df.to_csv("unload data/feature_importance_m21_4.csv", index=False)

In [ ]:
#Задаём группы признаков
structure_features = [
    'role_name',
    'schedule_id',
    'employment_id',
    'experience_ord'
]

region_features = [
    'economic_region',
    'region_name',
    'address.city_new'
]

geohash_features = [
    'geohash_4',
    'geohash_5',
    'geohash_6'
]

distance_features = [
    'distance_to_reg_center',
    'distance_missing'
]

coord_features = [
    'lat_sin',
    'lat_cos',
    'lon_sin',
    'lon_cos',
    'geo_available'
]

In [ ]:
#Считаем вклад групп
total_shap = shap_df['mean_abs_shap'].sum()

group_importance = {
    'Structure': shap_df.loc[
        shap_df.index.isin(structure_features),
        'mean_abs_shap'
    ].sum() / total_shap,

    'Region': shap_df.loc[
        shap_df.index.isin(region_features),
        'mean_abs_shap'
    ].sum() / total_shap,

    'Geo - Geohash': shap_df.loc[
        shap_df.index.isin(geohash_features),
        'mean_abs_shap'
    ].sum() / total_shap,

    'Geo - Distance': shap_df.loc[
        shap_df.index.isin(distance_features),
        'mean_abs_shap'
    ].sum() / total_shap,

    'Geo - Coordinates': shap_df.loc[
        shap_df.index.isin(coord_features),
        'mean_abs_shap'
    ].sum() / total_shap,
}

group_df = pd.DataFrame.from_dict(
    group_importance,
    orient='index',
    columns=['share']
).sort_values('share', ascending=False)

group_df

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=group_df.reset_index(),
    x='share',
    y='index',
    palette='Set2'
)

plt.title('Grouped SHAP Importance — Detailed Geo Split', fontsize=14)
plt.xlabel('Share of total SHAP')
plt.ylabel('')
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
colors = {
    'Structure': '#4C72B0',
    'Region': '#55A868',
    'Geo - Geohash': '#C44E52',
    'Geo - Distance': '#8172B3',
    'Geo - Coordinates': '#CCB974'
}

plt.figure(figsize=(8, 2))

left = 0

for group, value in group_importance.items():
    plt.barh(
        y='Model',
        width=value,
        left=left,
        color=colors[group],
        label=f"{group} ({value:.1%})"
    )
    left += value

plt.xlim(0, 1)
plt.title(f'Stacked SHAP Importance - {best_model_name}', fontsize=13)
plt.xlabel('Share of total SHAP')
plt.yticks([])
plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)
plt.tight_layout()
plt.show()

# Сравним метрики моделей этапов М1 и M2

In [ ]:
m1_cv_metrics = pd.read_csv("data_for_models/m1_cv_metrics.csv")
m2_cv_metrics = pd.read_csv("data_for_models/m2_cv_metrics.csv")

if "model" in m2_cv_metrics.columns:
    m2_cv_metrics = m2_cv_metrics.rename(columns={"model": "Model"})
# из M1 берем только OLS/Ridge CV-строки
m1_part = m1_cv_metrics.loc[
    m1_cv_metrics["Model"].isin([
        "OLS (sklearn pipeline, GroupKFold)",
        "Ridge (sklearn pipeline, GroupKFold)"
    ])
].copy()
m1_part["Stage"] = m1_part["Model"].map({
    "OLS (sklearn pipeline, GroupKFold)": "M1.1",
    "Ridge (sklearn pipeline, GroupKFold)": "M1.2"
})
m1_part["Algorithm"] = m1_part["Stage"].map({
    "M1.1": "OLS",
    "M1.2": "Ridge"
})
m1_part["Features"] = "Structural"
# из M2 берем baseline и лучшую гео-конфигурацию
m2_part = m2_cv_metrics.loc[
    m2_cv_metrics["Model"].isin(["M2.1.0_structure", "M2.1.4_distance"])
].copy()
m2_part["Stage"] = m2_part["Model"].map({
    "M2.1.0_structure": "M2.1.0",
    "M2.1.4_distance": "M2.1.4"
})
m2_part["Algorithm"] = "CatBoost"
m2_part["Features"] = m2_part["Stage"].map({
    "M2.1.0": "Structural",
    "M2.1.4": "Structural + Spatial"
})

# Приводим к единой схеме
common_cols = [
    "Stage", "Algorithm", "Features",
    "CV_RMSE_mean", "CV_RMSE_std",
    "CV_MAE_mean", "CV_MAE_std",
    "CV_R2_mean", "CV_R2_std",
    "n_folds"
]
m1_part = m1_part[common_cols].copy()
m2_part = m2_part[common_cols].copy()
final_results = pd.concat([m1_part, m2_part], ignore_index=True)
# Сортируем по этапу
stage_order = ["M1.1", "M1.2", "M2.1.0", "M2.1.4"]
final_results["Stage"] = pd.Categorical(final_results["Stage"], categories=stage_order, ordered=True)
final_results = final_results.sort_values("Stage").reset_index(drop=True)

final_results["CV_RMSE"] = final_results["CV_RMSE_mean"]
final_results["CV_R2"] = final_results["CV_R2_mean"]
display(final_results.round(6))

In [ ]:
sns.set(style="whitegrid", font_scale=1.1)
plot_df = final_results.copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# --- CV RMSE (mean) ---
sns.barplot(
    data=plot_df,
    x="Stage",
    y="CV_RMSE_mean",
    hue="Features",
    dodge=False,
    palette="Blues",
    ax=axes[0]
)
axes[0].set_title("CV RMSE by Stage")
axes[0].set_xlabel("Model Stage")
axes[0].set_ylabel("CV_RMSE_mean")
axes[0].legend(title="Features", loc="lower right")
# Подписи значений
for p in axes[0].patches:
    h = p.get_height()
    axes[0].annotate(
        f"{h:.4f}",
        (p.get_x() + p.get_width() / 2, h),
        ha="center",
        va="bottom",
        fontsize=9,
        xytext=(0, 3),
        textcoords="offset points"
    )
# --- CV R2 (mean) ---
sns.barplot(
    data=plot_df,
    x="Stage",
    y="CV_R2_mean",
    hue="Features",
    dodge=False,
    palette="Greens",
    ax=axes[1]
)
axes[1].set_title("CV R2 by Stage")
axes[1].set_xlabel("Model Stage")
axes[1].set_ylabel("CV_R2_mean")
axes[1].legend(title="Features", loc="lower right")
for p in axes[1].patches:
    h = p.get_height()
    axes[1].annotate(
        f"{h:.3f}",
        (p.get_x() + p.get_width() / 2, h),
        ha="center",
        va="bottom",
        fontsize=9,
        xytext=(0, 3),
        textcoords="offset points"
    )
plt.tight_layout()
plt.show()

In [ ]:
plot_df = final_results.copy()
plot_df = plot_df.set_index("Stage").loc[["M1.1", "M1.2", "M2.1.0", "M2.1.4"]].reset_index()
stages = [
    f"{row.Stage}\n{row.Algorithm}\n{row.Features.lower()}"
    for row in plot_df.itertuples(index=False)
]
rmse = plot_df["CV_RMSE_mean"].tolist()
r2 = plot_df["CV_R2_mean"].tolist()
sns.set(style="whitegrid", context="talk")
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
# R²
axes[0].plot(stages, r2, marker="o", linewidth=2.5)
axes[0].set_title("Model performance improvement (CV R²)")
axes[0].set_xlabel("Model stage")
axes[0].set_ylabel("CV R² mean")
for i, v in enumerate(r2):
    axes[0].text(i, v + 0.002, f"{v:.3f}", ha="center")
# RMSE
axes[1].plot(stages, rmse, marker="o", linewidth=2.5, color="orange")
axes[1].set_title("Model performance improvement (CV RMSE)")
axes[1].set_xlabel("Model stage")
axes[1].set_ylabel("CV RMSE mean")
for i, v in enumerate(rmse):
    axes[1].text(i, v + 0.001, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.show()